# PPO-GOMDP — Multi-Seed Training on Colab

Trains the PPO-GOMDP policy for the wildfire governance paper, running several seeds **in parallel**.

### Runtime
**Runtime → Change runtime type → T4 GPU + High-RAM.**

Pick the T4 runtime for its **vCPUs and RAM**, not its GPU — High-RAM shapes come with more cores, and cores are what this workload needs. Step 5 benchmarks both devices and picks the winner automatically.

### Why the GPU probably won't be used

Measured on Colab (grid 100 × 3000 steps, per episode):

| | rollout | update | total |
|---|---|---|---|
| CPU | 5.82 s | 1.45 s | **7.27 s** |
| CUDA | 6.05 s | 1.49 s | 7.54 s |

The GPU lands at parity. Rollout is **batch-1**: each step needs one kernel launch plus a device sync to read the sampled action, and that round trip costs about as much as doing the 2.6 M-parameter matmul on CPU outright. The T4 would only pay off if the policy forward were batched across seeds, which this trainer deliberately does not do — each seed is an independent process, which is simpler and keeps every run on the exact code path used for evaluation.

So: **rollout runs on CPU, and throughput comes from parallel workers.** The GPU sits idle by design.

> An earlier version of this notebook was ~2× *slower* on GPU (38.35 s/episode). The cause was `select_actions` sampling each UAV head separately with `.item()` — 20 device syncs per step, 60 000 per episode. Fusing the 20 heads into one matmul and batching the sample to a single sync cut CPU time from 19.90 s to 7.27 s per episode, a **2.7× speedup**. That fix, not the accelerator, is where the performance came from.

### Before you run
This notebook clones from GitHub, so **push your local changes first**. Step 2 verifies the clone contains the fused-head policy and aborts if not — an older commit would train ~2.7× slower.

## Step 1 — Report the runtime

A GPU is *not* required — Step 5 will most likely select CPU. What matters is **vCPU count**, since that sets how many seeds train concurrently.

In [ ]:
import subprocess, sys, os, multiprocessing
import torch

nvsmi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvsmi.stdout or "(no nvidia-smi — CPU-only runtime)")

cpus = multiprocessing.cpu_count()
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9

print(f"torch   : {torch.__version__}  (CUDA {torch.version.cuda})")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  |  VRAM {props.total_memory / 1e9:.1f} GB")
else:
    # Not fatal: rollout is batch-1 and runs on CPU anyway (see header).
    print("GPU     : none available — fine, CPU is the faster path here")
print(f"CPU     : {cpus} vCPUs  |  RAM {ram:.1f} GB   <- the binding constraint")

if cpus < 4:
    print(f"\n[WARN] Only {cpus} vCPUs. Seeds train concurrently one-per-core, so this")
    print("       runtime will be slow. A High-RAM shape usually provides 8.")
if ram < 30:
    print("\n[WARN] Standard-RAM runtime. Each worker needs ~400 MB; High-RAM is")
    print("       recommended but you can proceed with fewer workers.")

## Step 2 — Clone the repository

For a **private** repo, replace the URL with a token form:
`https://<GITHUB_TOKEN>@github.com/aliakarma/wildfire-governance-agentic-ai.git`
(use a fine-grained read-only token, and clear the cell output afterwards).

In [ ]:
REPO_URL = "https://github.com/aliakarma/wildfire-governance-agentic-ai.git"
BRANCH   = "main"
REPO_DIR = "/content/wildfire-governance-agentic-ai"

import os, shutil, subprocess
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
    check=True,
)
os.chdir(REPO_DIR)

sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print(f"\ncloned {BRANCH} @ {sha} -> {REPO_DIR}")

# Verify the clone has the performance-critical code. The fused policy head is
# what makes an episode 7.3s instead of 19.9s, so training an older commit would
# waste ~2.7x the wall-clock for identical results.
agent_src = open("src/wildfire_governance/rl/ppo_agent.py").read()
checks = {
    "experiments/11c_train_multiseed.py": os.path.exists("experiments/11c_train_multiseed.py"),
    "requirements-colab.txt":             os.path.exists("requirements-colab.txt"),
    "device= support in ppo_agent":       "self.device" in agent_src,
    "fused policy head (perf critical)":  "self.head = nn.Linear" in agent_src,
}
for name, ok in checks.items():
    print(f"  [{'OK ' if ok else 'MISSING'}] {name}")

if not all(checks.values()):
    raise SystemExit(
        "\nThis commit predates the GPU/perf work. Push your local changes "
        "and re-run this cell."
    )
print("\nall checks passed.")

## Step 3 — Install dependencies

Uses `requirements-colab.txt`, which omits torch (Colab's CUDA build is kept) and the geospatial stack (only needed for VIIRS preprocessing). Takes well under a minute; installing the full `requirements.txt` would take many minutes and can downgrade torch to a CPU build.

In [ ]:
!pip install -q -r requirements-colab.txt

import sys, os
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = f"{REPO_DIR}/src:{REPO_DIR}"

import torch
print(f"torch {torch.__version__}  cuda_available={torch.cuda.is_available()}")
if not torch.cuda.is_available():
    # Only a warning: CPU is the faster path for batch-1 rollout regardless.
    print("[NOTE] No CUDA. Training will run on CPU, which is expected here.")

## Step 4 — Sanity check

Runs the test suite and a 2-seed smoke train. If either fails, stop here — a long run would only waste GPU hours.

In [ ]:
!python -m pytest tests -q 2>&1 | tail -5
print("\n--- smoke train (2 seeds x 3 episodes) ---")
!python experiments/11c_train_multiseed.py --smoke 2>&1 | grep -v '^{' | tail -12

## Step 5 — Benchmark GPU vs CPU

Times one real episode on each device so the schedule below is based on your actual runtime, not an assumption. Takes ~1 minute.

In [ ]:
import time, numpy as np, torch
from wildfire_governance.rl.gomdp_env import GOMMDPGymEnv
from wildfire_governance.rl.ppo_agent import PPOGOMDPAgent
from wildfire_governance.simulation.grid_environment import EnvironmentConfig

def time_episode(device, grid=100, steps=3000, n_uavs=20):
    env = GOMMDPGymEnv(config=EnvironmentConfig(grid_size=grid, n_timesteps=steps),
                       n_uavs=n_uavs, enable_governance=True)
    agent = PPOGOMDPAgent(grid_size=grid, n_uavs=n_uavs, device=device)
    obs, _ = env.reset(seed=0)
    O, A, R, D = [], [], [], []
    t0 = time.time(); done = False
    while not done:
        ad = agent.select_actions(obs, env._fleet)
        arr = np.array([ad.get(i, 0) for i in range(n_uavs)])
        nobs, r, term, trunc, _ = env.step(arr)
        O.append(obs.copy()); A.append(ad); R.append(float(r)); D.append(term or trunc)
        obs = nobs; done = term or trunc
    roll = time.time() - t0
    t0 = time.time(); agent.update(O, A, R, D); upd = time.time() - t0
    return roll, upd

devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
results = {}
for dev in devices:
    roll, upd = time_episode(dev)
    results[dev] = roll + upd
    print(f"{dev:5s}: rollout {roll:6.2f}s + update {upd:5.2f}s = {roll+upd:6.2f}s/episode")

# Use whichever device actually won, with a margin so a marginal GPU edge does
# not buy GPU contention between parallel workers for nothing.
#
# Expect CPU to win. Rollout is batch-1: a kernel launch plus the device sync
# needed to read the sampled action costs about as much as doing the 2.6M-param
# matmul on CPU outright. The T4 would only pay off if the policy forward were
# batched across seeds, which this trainer does not do (each seed is an
# independent process).
if "cuda" in results:
    speedup = results["cpu"] / results["cuda"]
    print(f"\nGPU speedup: {speedup:.2f}x")
    DEVICE = "cuda" if speedup > 1.15 else "cpu"
else:
    DEVICE = "cpu"

SEC_PER_EPISODE = min(results.values())
print(f"\nselected device : {DEVICE}   ({SEC_PER_EPISODE:.2f}s/episode)")
if DEVICE == "cpu" and "cuda" in results:
    print("-> No useful GPU speedup at batch size 1; running on CPU and scaling")
    print("   with parallel workers instead. The GPU stays idle by design.")

## Step 6 — Configure the run

`N_SEEDS` is how many independent policies to train (the paper's validation curve is a mean ± SD across seeds). `WORKERS` is how many run concurrently.

**Sizing.** Since rollout runs on CPU, the binding constraint is **vCPU count**, not VRAM. Each worker is pinned to one torch thread (`torch.set_num_threads(1)`) and needs roughly one core plus ~400 MB RAM, so `WORKERS = min(N_SEEDS, vCPUs)` is the right default — on a High-RAM runtime, RAM is nowhere near the limit.

If Step 5 did select `cuda`, cap workers at ~6 to limit GPU contention.

In [ ]:
import multiprocessing

N_SEEDS  = 5      # independent policies; paper's curve uses 5 held-out seeds
EPISODES = 1000   # per seed

vcpus = multiprocessing.cpu_count()
# CPU rollout scales with cores; a GPU run needs a cap to limit contention.
WORKERS = min(N_SEEDS, vcpus) if DEVICE == "cpu" else min(N_SEEDS, vcpus, 6)

waves = -(-N_SEEDS // WORKERS)  # ceiling division
# Workers share memory bandwidth, so per-episode time degrades somewhat when
# every core is busy. Bracket the estimate rather than quoting a single number.
low_h  = SEC_PER_EPISODE * EPISODES * waves / 3600
high_h = low_h * 1.35

print(f"device={DEVICE}  seeds={N_SEEDS}  episodes={EPISODES}")
print(f"vCPUs={vcpus}  ->  workers={WORKERS}  ({waves} wave(s))")
print(f"~{SEC_PER_EPISODE:.2f}s/episode uncontended")
print(f"\nestimated wall-clock: {low_h:.1f}-{high_h:.1f} h")

if high_h > 10:
    print("\n[WARN] May exceed a typical Colab session. Reduce EPISODES or N_SEEDS,")
    print("       and keep Drive output (Step 7) enabled so a disconnect is recoverable.")
if N_SEEDS > vcpus:
    print(f"\n[NOTE] {N_SEEDS} seeds on {vcpus} vCPUs runs in {waves} sequential waves.")
    print(f"       Setting N_SEEDS={vcpus} would finish in one wave for the same wall-clock.")

## Step 7 — (Recommended) Persist results to Google Drive

Colab disconnects lose `/content`. Writing results to Drive means a dropped session costs only the episodes since the last write — the trainer appends its learning curve after **every** episode, so nothing else is lost.

Skip this cell if you'd rather keep everything local and download at the end.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/gomdp_training/multiseed"
else:
    OUTDIR = f"{REPO_DIR}/results/runs/multiseed_colab"

import os
os.makedirs(OUTDIR, exist_ok=True)
print(f"results -> {OUTDIR}")

## Step 8 — Train

Launches in the background so the next cell can show live progress. Each seed appends to its own `ppo_learning_curve.csv` every episode and rewrites `status.json`, so progress is always readable and an interrupted run still leaves usable history.

In [ ]:
import subprocess, os

LOG = "/content/train.log"
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src:{REPO_DIR}")
cmd = [
    "python", "experiments/11c_train_multiseed.py",
    "--seeds", str(N_SEEDS), "--episodes", str(EPISODES),
    "--workers", str(WORKERS), "--device", DEVICE, "--outdir", OUTDIR,
]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                        stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print(f"launched pid={proc.pid}; log -> {LOG}")

## Step 9 — Monitor

Re-runnable at any time. Polls each seed's `status.json`. Interrupting this cell does **not** stop training.

In [ ]:
import json, time, glob, os
from IPython.display import clear_output

POLL_S = 30
try:
    while True:
        clear_output(wait=True)
        stats = []
        for p in sorted(glob.glob(f"{OUTDIR}/seed_*/status.json")):
            try:
                stats.append(json.load(open(p)))
            except (json.JSONDecodeError, OSError):
                pass  # mid-write; skip this tick
        if not stats:
            print("waiting for first episode...")
        else:
            print(f"{'seed':>5} {'episode':>12} {'pct':>7} {'best_rw':>10} {'ETA':>10}")
            for s in stats:
                eta = f"{s['eta_s']/60:.0f} min" if s["eta_s"] < 5400 else f"{s['eta_s']/3600:.1f} h"
                print(f"{s['seed']:>5} {s['episode']:>6}/{s['n_episodes']:<5} "
                      f"{s['pct']:>6.1f}% {s['best_reward']:>10.1f} {eta:>10}")
            print(f"\noverall: {sum(x['pct'] for x in stats)/len(stats):.1f}%")
        if proc.poll() is not None:
            print(f"\nTRAINING FINISHED (exit {proc.returncode})")
            break
        time.sleep(POLL_S)
except KeyboardInterrupt:
    print("\n(monitor stopped; training continues in background)")

## Step 10 — Learning curve

Plots validation $L_d$ (mean ± SD across seeds), the figure the paper's training section needs. Reads the aggregate if training finished, otherwise builds it from whatever episodes exist so far.

In [ ]:
import pandas as pd, numpy as np, glob
import matplotlib.pyplot as plt

frames = []
for p in sorted(glob.glob(f"{OUTDIR}/seed_*/ppo_learning_curve.csv")):
    df = pd.read_csv(p)
    df["seed"] = int(p.split("seed_")[1].split("/")[0])
    frames.append(df)

allc = pd.concat(frames, ignore_index=True)
agg = allc.groupby("episode").agg(
    ld_mean=("ld", "mean"), ld_std=("ld", "std"),
    reward_mean=("reward", "mean"),
    compliance_mean=("compliance", "mean"),
).reset_index()

# Smooth for readability; the CSV keeps the raw per-episode values.
W = max(1, len(agg) // 50)
sm = agg.rolling(W, min_periods=1).mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sm["episode"], sm["ld_mean"], color="tab:blue")
ax[0].fill_between(sm["episode"], sm["ld_mean"] - sm["ld_std"],
                   sm["ld_mean"] + sm["ld_std"], alpha=0.2, color="tab:blue")
ax[0].set_xlabel("Training episode"); ax[0].set_ylabel("Validation $L_d$ (steps)")
ax[0].set_title(f"PPO-GOMDP learning curve ({allc['seed'].nunique()} seeds)")
ax[0].grid(alpha=0.3)

ax[1].plot(sm["episode"], sm["reward_mean"], color="tab:orange")
ax[1].set_xlabel("Training episode"); ax[1].set_ylabel("Episode reward")
ax[1].set_title("Reward"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

tail = agg.tail(50)
print(f"episodes completed : {len(agg)}")
print(f"final L_d (last 50): {tail['ld_mean'].mean():.2f} +/- {tail['ld_std'].mean():.2f}")
print(f"governance compliance: {100*agg['compliance_mean'].mean():.1f}%   <- Theorem 1")
agg.to_csv(f"{OUTDIR}/learning_curve_aggregate.csv", index=False)

## Step 11 — Export

Packages checkpoints, curves and summary for download. Copy `best_checkpoint.pt` into `src/wildfire_governance/rl/checkpoints/ppo_gomdp_best.pt` locally to use it for evaluation.

In [ ]:
import shutil, json, os, glob

summary_path = f"{OUTDIR}/summary.json"
if os.path.exists(summary_path):
    print(json.dumps(json.load(open(summary_path)), indent=2)[:1200])

archive = shutil.make_archive("/content/gomdp_training_results", "zip", OUTDIR)
print(f"\n{archive}  ({os.path.getsize(archive)/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"(auto-download unavailable: {exc}; results are in {OUTDIR})")